# AIHub T자형 교차로 데이터 점검 01 — 영상 목록 만들기

이 노트북의 목표는 중복 제거가 아니라 **원본 1,059개 영상을 수정하지 않고 기본 정보를 `inventory.csv`로 저장하는 것**입니다.

이번 단계에서 확인하는 항목:
- 파일명과 경로
- 파일 크기
- 프레임 수와 FPS
- 영상 길이와 해상도
- OpenCV로 정상적으로 읽을 수 있는지

밝기 중복 탐지는 이 결과를 확인한 다음 별도 단계로 추가합니다.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

VIDEO_ROOT = Path(
    '/mnt/c/ai허브/T자형교차로/095.교통사고_영상_데이터/'
    '01.데이터/1.Training/원천데이터_231108_add/'
    'TS_차대차_영상_T자형교차로'
)

REPO_ROOT = Path('/home/sungmin/blackbox-dacon')
OUTPUT_DIR = REPO_ROOT / 'data/stage2/aihub_tjunction/metadata'
INVENTORY_PATH = OUTPUT_DIR / 'inventory.csv'

assert VIDEO_ROOT.is_dir(), f'영상 폴더를 찾을 수 없습니다: {VIDEO_ROOT}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('영상 폴더:', VIDEO_ROOT)
print('결과 폴더:', OUTPUT_DIR)

In [ ]:
VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv'}

video_paths = sorted(
    path for path in VIDEO_ROOT.rglob('*')
    if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS
)

print(f'발견한 영상: {len(video_paths):,}개')
print('파일명 예시:')
for path in video_paths[:10]:
    print('-', path.name)

In [ ]:
def read_video_metadata(path: Path) -> dict:
    capture = cv2.VideoCapture(str(path))
    opened = capture.isOpened()

    if opened:
        frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = float(capture.get(cv2.CAP_PROP_FPS))
        width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    else:
        frame_count, fps, width, height = 0, 0.0, 0, 0

    capture.release()
    duration_sec = frame_count / fps if fps > 0 else np.nan

    return {
        'video_id': path.stem,
        'file_name': path.name,
        'relative_path': path.relative_to(VIDEO_ROOT).as_posix(),
        'size_mb': path.stat().st_size / (1024 ** 2),
        'frame_count': frame_count,
        'fps': fps,
        'duration_sec': duration_sec,
        'width': width,
        'height': height,
        'read_ok': opened and frame_count > 0 and fps > 0,
    }


rows = [read_video_metadata(path) for path in tqdm(video_paths, desc='영상 정보 확인')]
inventory = pd.DataFrame(rows)
inventory.to_csv(INVENTORY_PATH, index=False, encoding='utf-8-sig')

print('저장 완료:', INVENTORY_PATH)
display(inventory.head())

In [ ]:
summary = pd.Series({
    '전체 영상 수': len(inventory),
    '정상 읽기': int(inventory['read_ok'].sum()),
    '읽기 실패': int((~inventory['read_ok']).sum()),
    '총 용량(GB)': inventory['size_mb'].sum() / 1024,
    '영상 길이 중앙값(초)': inventory['duration_sec'].median(),
    '프레임 수 중앙값': inventory['frame_count'].median(),
})
display(summary.to_frame('값'))

print('FPS 분포')
display(inventory['fps'].round(3).value_counts(dropna=False).sort_index().to_frame('영상 수'))

print('해상도 분포')
display(
    inventory.value_counts(['width', 'height'])
    .rename('영상 수')
    .reset_index()
    .head(20)
)

## 선택 확인: 무작위 영상의 가운데 프레임 보기

데이터의 전반적인 화면 구도와 밝기 변형 형태를 빠르게 확인합니다. 이 셀은 파일을 변경하지 않습니다.

In [ ]:
def read_middle_frame(path: Path):
    capture = cv2.VideoCapture(str(path))
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    capture.set(cv2.CAP_PROP_POS_FRAMES, max(0, frame_count // 2))
    ok, frame = capture.read()
    capture.release()
    if not ok:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


sample = inventory[inventory['read_ok']].sample(12, random_state=42)
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
for axis, row in zip(axes.flat, sample.itertuples(index=False)):
    frame = read_middle_frame(VIDEO_ROOT / row.relative_path)
    if frame is not None:
        axis.imshow(frame)
    axis.set_title(row.file_name, fontsize=8)
    axis.axis('off')
plt.tight_layout()
plt.show()

## 이번 노트북의 종료 지점

`inventory.csv`가 정상적으로 저장되고 요약 결과를 확인하면 이번 단계는 끝입니다. 다음 단계에서 밝기를 정규화한 여러 프레임 지문을 만들고, 동일 영상 후보를 그룹화합니다.